In [ ]:
import argparse
import json
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, pipeline,
)
import sys, os
sys.path.insert(0, '/content/drive/MyDrive/Thesis/models')

from utils import DATA_DIR, MODELS_DIR, RESULTS_DIR, EKMAN_LABELS, set_all_seeds
from evaluate import compute_metrics, save_metrics, plot_confusion_matrix


In [ ]:
SARCASTIC_INVERSION = {
    "joy":      "sadness",
    "sadness":  "sadness",
    "anger":    "anger",
    "disgust":  "disgust",
    "fear":     "fear",
    "surprise": "surprise",
    "neutral":  "neutral",
}


In [ ]:
def load_phase4_winner():
    path = (
        MODELS_DIR
        / "phase4_runs"
        / "focal_loss_runs"
        / "phase4_focal_g1_seed42_correct"
        / "best_model"
    )

    tok = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
    mod = AutoModelForSequenceClassification.from_pretrained(
        str(path),
        local_files_only=True,
    )

    mod.eval()
    if torch.cuda.is_available():
        mod = mod.cuda()

    return tok, mod

def load_sarcasm_detector():
    return pipeline(
        "text-classification",
        model="cardiffnlp/twitter-roberta-base-irony",
        device=0 if torch.cuda.is_available() else -1,
        top_k=None,
    )


def predict_emotion_batch(texts, tok, mod, batch_size=32):
    all_preds, all_probs = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=128)
        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}
        with torch.no_grad():
            logits = mod(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
    return all_preds, all_probs


def predict_sarcasm_batch(texts, detector, batch_size=32):
    scores = []
    for i in range(0, len(texts), batch_size):
        out = detector(texts[i : i + batch_size])
        for entry in out:
            irony_score = next(d["score"] for d in entry
                                if d["label"] == "irony")
            scores.append(irony_score)
    return scores


def sarcasm_aware_predict(texts, threshold=0.7):
    tok, mod = load_phase4_winner()
    detector = load_sarcasm_detector()

    raw_preds, _ = predict_emotion_batch(texts, tok, mod)
    irony_scores = predict_sarcasm_batch(texts, detector)

    final_preds = []
    sarcasm_flags = []
    for raw_idx, irony in zip(raw_preds, irony_scores):
        raw_label = EKMAN_LABELS[raw_idx]
        if irony >= threshold:
            final_label = SARCASTIC_INVERSION[raw_label]
            sarcasm_flags.append(True)
        else:
            final_label = raw_label
            sarcasm_flags.append(False)
        final_preds.append(EKMAN_LABELS.index(final_label))
    return final_preds, sarcasm_flags, irony_scores

In [ ]:
def evaluate_on_test(threshold=0.7, seed=42):
    set_all_seeds(seed)
    test_df = pd.read_parquet(DATA_DIR / "test_single_light.parquet")
    texts = test_df["text_clean"].tolist()
    y_true = test_df["ekman_id"].values

    y_pred, flags, irony_scores = sarcasm_aware_predict(texts, threshold=threshold)
    n_flagged = sum(flags)
    print(f"Sarcasm detector fired on {n_flagged}/{len(texts)} test inputs "
          f"({100 * n_flagged / len(texts):.1f}%).")

    metrics = compute_metrics(y_true, y_pred, EKMAN_LABELS)
    metrics["sarcasm_threshold"] = threshold
    metrics["n_flagged_sarcastic"] = int(n_flagged)
    metrics["n_total"] = int(len(texts))

    run_name = f"phase5_sarcasm_aware_t{threshold}"
    save_metrics(metrics, run_name)
    plot_confusion_matrix(y_true, y_pred, EKMAN_LABELS, run_name)
    print(f"Macro-F1 with sarcasm gating: {metrics['macro_f1']:.4f}")
    return metrics

In [ ]:
QUALITATIVE_DEMO = [
    "I am so happy, my dog just died.",
    "Oh great, another Monday. Just what I needed.",
    "Wow, you're so smart. /s",
    "Yeah, the food was just amazing.",        # could be sarcastic OR genuine
    "Thank you so much, this really made my day.",
    "I love waiting in traffic for an hour.",
    "The funeral was beautiful, I cried.",
    "Fantastic, my car broke down again.",
    "I am so happy to be alive"
]


def run_qualitative(threshold=0.7):
    preds, flags, irony_scores = sarcasm_aware_predict(QUALITATIVE_DEMO,
                                                        threshold=threshold)
    print(f"\nQualitative comparison (threshold={threshold}):")
    print(f"{'Input':<50} {'Irony':>6} {'Final':<10} {'Flag':<5}")
    print("-" * 80)
    for txt, p, flg, ir in zip(QUALITATIVE_DEMO, preds, flags, irony_scores):
        short = txt[:47] + "..." if len(txt) > 50 else txt
        print(f"{short:<50} {ir:>6.2f} {EKMAN_LABELS[p]:<10} "
              f"{'YES' if flg else 'no':<5}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--threshold", type=float, default=0.7)
    parser.add_argument("--qualitative-only", action="store_true")
    args = parser.parse_args([]) # Modified to handle Colab environment

    run_qualitative(threshold=args.threshold)
    if not args.qualitative_only:
        evaluate_on_test(threshold=args.threshold)


Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...


Qualitative comparison (threshold=0.7):
Input                                               Irony Final      Flag 
--------------------------------------------------------------------------------
I am so happy, my dog just died.                     0.73 sadness    YES  
Oh great, another Monday. Just what I needed.        0.99 sadness    YES  
Wow, you're so smart. /s                             0.94 sadness    YES  
Yeah, the food was just amazing.                     0.86 sadness    YES  
Thank you so much, this really made my day.          0.64 joy        no   
I love waiting in traffic for an hour.               0.99 sadness    YES  
The funeral was beautiful, I cried.                  0.65 joy        no   
Fantastic, my car broke down again.                  0.95 sadness    YES  
I am so happy to be alive                            0.62 joy        no   


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Sarcasm detector fired on 797/4590 test inputs (17.4%).
Saved metrics -> /content/drive/MyDrive/Thesis/results/phase5_sarcasm_aware_t0.7.json
Saved figure -> /content/drive/MyDrive/Thesis/figures/cm_phase5_sarcasm_aware_t0.7.png
Macro-F1 with sarcasm gating: 0.5760


In [ ]:
text = "I love this"
print(f"INPUT: {text}\n")

# Corrected way to get sarcasm-aware prediction for a single text
preds, flags, irony_scores = sarcasm_aware_predict([text])

# Display the results for the single text
print(f"{'Input':<50} {'Irony':>8} {'Final':<10} {'Flag':<7}")
print("-" * 80)
short = text[:47] + "..." if len(text) > 50 else text
print(f"{short:<50} {irony_scores[0]:>6.2f} {EKMAN_LABELS[preds[0]]:<10} "
      f"{'YES' if flags[0] else 'no':<5}")

# Removed incorrect calls:
# print(run_qualitative(text))
# evaluate_on_test(text)

INPUT: I love this



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Input                                                 Irony Final      Flag   
--------------------------------------------------------------------------------
I love this                                          0.91 sadness    YES  
